In [1]:
"""
Name: Jacob Sharon
Date: 11-09-2026
Class: CSC635

This file implements the Feed Forward Neural Network for our groups class project. 

Updates: 
11-09-2026: 
    - Imports, rough outline as TODO cells, set up MLP class for main experiment. 
11-18-2026: 
    - Began preprocessing. 
"""

'\nName: Jacob Sharon\nDate: 11-09-2026\nClass: CSC635\n\nThis file implements the Feed Forward Neural Network for our groups class project. \n\nUpdates: \n11-09-2026: \n    - Imports, rough outline as TODO cells, set up MLP class for main experiment. \n11-18-2026: \n    - Imported primary dataset anmd \n'

In [2]:
# Imports

# Data and Preprocessing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Pytorch 
import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import TensorDataset, DataLoader

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

In [3]:
# Load the dataset

"""
TODO: 
    - The dataset is split between two files. Join where appropriate.
"""

diabetic_data = pd.read_csv("../Data/diabetic_data.csv")

diabetic_data.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [64]:
# Preprocessing: Discovery and Feature Selection

"""
For your knowledge, I am a Medical Laboratory Scientist and perform high complexity on patients of this demographic. My work is primarily 
in clinical chemistry and my previous experience includes supervisory/managerial roles including billing and ordering using the coding 
system described in this dataset. I am basing decisions about feature selection on my professional knowlege and understanding of the problem 
being solved. 

Based on review of the CSV: 

Attributes with a lot of missing values: 
    - weight: categrocial, weight in pounds
    - payer_code: categorical, int mapping to the payer (e.g. self, blue cross, anthem, etc...)
    - medical_specialty: categorical, int mapping to speciality of admitting provider

Attributes with inconsistent data: 
    - diag_1: categorical, first 3 digits of ICD9 (billing diagnistic code)
        - some only 1 digit, 
        - some decimal, 
        - some included the letter
        
    - diag_2: categorical, secondary diagnosis code
    - diag_3: categorical, tertiary diagnosis code
        - diag_2 and diag_3 are sometimes duplicated
        - some patients do not have more than one diagnostic code

Others:
    - max_glu_serum: categorical, indicates the range of the result or if the test was not taken. 
        - Values: [>200, >300, normal, and none if not measured]
        - if not measured on most patients it is not very helpful

    - A1Cresult: categorical, range of the result or if the test was not taken.
        - Values: [>8 if the result was greater than 8%, 
                   >7 if the result was greater than 7% but less than 8%, 
                   normal if the result was less than 7%,
                   none if not measured.]
        - Should be recatogeorized to [High (>7), norm, and none] due to a Hgb A1C > 6.5% being diagnostic for diabetes
        - Increased range above what is clinically interpreted as high is not helpful

    - {drug}: categorical, The feature indicates whether the drug was prescribed or there was a change in the dosage.
        - Values: [up if the dosage was increased during the encounter, 
                   down if the dosage was decreased, 
                   steady if the dosage did not change, 
                   no if the drug was not prescribed]
        - Need to get counts of patients on each drug, if only small percentage are prescribed then will not help model generalize.
"""

# Step 1: Get counts of missing values on weight, payer_code, medical_specialty. The missing values are ? in the csv.

print("1. Counts of missing values")
print(diabetic_data[['weight', 'payer_code', 'medical_specialty']].describe(), "\n")

# Step 2: Get counts of diagnostic codes where it is not a 3 digit int, may be able to drop those rows.
diag_cols = ['diag_1', 'diag_2', 'diag_3']

print("2. Diagnostic codes that are not 3 digit integers")
for col in diag_cols: 
    print(col, (~diabetic_data[col].astype(str).str.fullmatch(r'\d{3}')).sum())

# Step 3: Get counts of max_glu_serum
print("\n3. Count of max_glu_serum with a value")
print(diabetic_data['max_glu_serum'].count())

# Step 4: Get details about A1C and recategorize values. 
print("\n4. Count of >7 or >8 A1C values that were changed to 'High'")
diabetic_data['A1Cresult'] = diabetic_data['A1Cresult'].replace(['>8', '>7'], 'High')
replaced_values = diabetic_data[diabetic_data['A1Cresult'].isin(['High'])]
print(len(replaced_values))

# Step 5: Get counts of patients on each drug. 
# TODO - get counts of each drug and determine if will be enough to be useful for model. 

1. Counts of missing values
        weight payer_code medical_specialty
count   101766     101766            101766
unique      10         18                73
top          ?          ?                 ?
freq     98569      40256             49949 

2. Diagnostic codes that are not 3 digit integers
diag_1 12834
diag_2 11177
diag_3 13610

3. Count of max_glu_serum with a value
5346

4. Count of >7 or >8 A1C values that were changed to 'High'
12028


In [ ]:
# Neural Network Implementation

class MLP(nn.Module):
    """
    A fully connected feedforward neural network (Multi-Layer Perceptron)
    used for binary classification of early readmission of diabteic patients
    within 30 days of discharge.

    This implementation supports an arbitrary number of hidden layers, each
    followed by:
      • Batch Normalization (except after the first layer)
      • Dropout (p = 0.02)
      • ReLU activation

    Kaiming uniform initialization is applied to all Linear layer weights,
    and biases are initialized to zero. The final layer outputs an unbounded
    logit suitable for use with BCEWithLogitsLoss.

    Parameters
    ----------
    input_size : int
        Number of input features.

    hidden_sizes : list of int
        A list specifying the width of each hidden layer.
        Example: [64, 32] produces a network with two hidden layers.

    output_size : int
        Number of output neurons. For binary classification, this is 1.

    Attributes
    ----------
    network : nn.Sequential
        The sequential model containing all layers in forward order.
    """

    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()

        layers = []

        # First hidden layer (no batch norm / dropout here)
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())

        # Remaining hidden layers
        for i in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))
            layers.append(nn.BatchNorm1d(hidden_sizes[i + 1]))
            layers.append(nn.Dropout(p=0.02))
            layers.append(nn.ReLU())

        # Output layer (produces logits)
        layers.append(nn.Linear(hidden_sizes[-1], output_size))

        # Construct sequential model
        self.network = nn.Sequential(*layers)

        # Initialize parameters
        self._init_weights()

    def _init_weights(self):
        """
        Initialize weights of all Linear layers using Kaiming uniform
        initialization, which is recommended for ReLU-activated networks.

        Bias terms are initialized to zero.
        """
        for layer in self.network:
            if isinstance(layer, nn.Linear):
                init.kaiming_uniform_(layer.weight, nonlinearity='relu')
                if layer.bias is not None:
                    init.zeros_(layer.bias)

    def forward(self, x):
        """
        Perform a forward pass through the network.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (batch_size, input_size).

        Returns
        -------
        torch.Tensor
            Output logits of shape (batch_size, output_size).
            These logits should be passed to sigmoid() or directly to
            BCEWithLogitsLoss for stable binary classification.
        """
        return self.network(x)

In [ ]:
# Experimental Designs and Hyperparameter Tuning

"""
TODO: 
    - Begin with using all input features and then perform feature selection to narrow. 
    - Define network architectures to test, probably keep limited to two hidden layers for this class.
    - Decide which metrics to collect and best representation. 
"""

In [ ]:
# Data Collection and Plotting